# 📊 Bayesian Network — HillClimb Pipeline
**Dataset** : `child.bif` (simulé)  
**Objectif** : Apprendre la structure et les paramètres d'un réseau à partir de données, puis réaliser des inférences diagnostiques et prédictives.


## 1. Setup & Préparation des données

In [16]:
%pip install pgmpy pandas numpy scikit-learn -q

import warnings
warnings.filterwarnings('ignore')
from pgmpy import global_vars
global_vars.SHOW_PROGRESS = False

import pandas as pd
import numpy as np
import networkx as nx

from pgmpy.readwrite import BIFReader
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

# Charger le réseau gold depuis le fichier .bif
reader = BIFReader("child.bif")
gold_model = reader.get_model()

print(f" Modèle gold chargé")
print(f"   Nœuds ({len(gold_model.nodes())}) : {list(gold_model.nodes())}")
print(f"   Arcs  ({len(gold_model.edges())}) : {list(gold_model.edges())}")
print(f"   États de 'Disease' : {gold_model.get_cpds('Disease').state_names['Disease']}")

Note: you may need to restart the kernel to use updated packages.
 Modèle gold chargé
   Nœuds (20) : ['BirthAsphyxia', 'HypDistrib', 'HypoxiaInO2', 'CO2', 'ChestXray', 'Grunting', 'LVHreport', 'LowerBodyO2', 'RUQO2', 'CO2Report', 'XrayReport', 'Disease', 'GruntingReport', 'Age', 'LVH', 'DuctFlow', 'CardiacMixing', 'LungParench', 'LungFlow', 'Sick']
   Arcs  (25) : [('BirthAsphyxia', 'Disease'), ('HypDistrib', 'LowerBodyO2'), ('HypoxiaInO2', 'LowerBodyO2'), ('HypoxiaInO2', 'RUQO2'), ('CO2', 'CO2Report'), ('ChestXray', 'XrayReport'), ('Grunting', 'GruntingReport'), ('Disease', 'Age'), ('Disease', 'LVH'), ('Disease', 'DuctFlow'), ('Disease', 'CardiacMixing'), ('Disease', 'LungParench'), ('Disease', 'LungFlow'), ('Disease', 'Sick'), ('LVH', 'LVHreport'), ('DuctFlow', 'HypDistrib'), ('CardiacMixing', 'HypDistrib'), ('CardiacMixing', 'HypoxiaInO2'), ('LungParench', 'HypoxiaInO2'), ('LungParench', 'CO2'), ('LungParench', 'ChestXray'), ('LungParench', 'Grunting'), ('LungFlow', 'ChestXray'), (

## 2. Simulation de données depuis le modèle gold

In [17]:
N_SAMPLES = 5000
SEED = 42

# Générer N observations cohérentes avec les CPDs du réseau gold
df = gold_model.simulate(n_samples=N_SAMPLES, seed=SEED, show_progress=False)

print(f" {N_SAMPLES} observations simulées")
print(f"   Shape : {df.shape}")
print()
print("Distribution de 'Disease' :")
print(df['Disease'].value_counts(normalize=True).round(3))
df.head()

 5000 observations simulées
   Shape : (5000, 20)

Distribution de 'Disease' :
Disease
TGA       0.341
Fallot    0.302
PAIVS     0.212
TAPVD     0.053
PFC       0.048
Lung      0.044
Name: proportion, dtype: float64


,LowerBodyO2,LungFlow,ChestXray,XrayReport,BirthAsphyxia,Grunting,LungParench,DuctFlow,LVH,CO2,Sick,CO2Report,LVHreport,RUQO2,HypoxiaInO2,Age,GruntingReport,HypDistrib,CardiacMixing,Disease
0,5-12,Low,Oligaemic,Oligaemic,no,no,Normal,Lt_to_Rt,no,Low,no,>=7.5,no,5-12,Moderate,0-3_days,no,Equal,Complete,Fallot
1,5-12,Low,Oligaemic,Normal,no,no,Normal,Lt_to_Rt,no,High,yes,<7.5,no,<5,Moderate,0-3_days,yes,Equal,Mild,Fallot
2,5-12,Low,Oligaemic,Oligaemic,no,no,Normal,Lt_to_Rt,yes,Normal,yes,<7.5,yes,5-12,Severe,0-3_days,no,Equal,Complete,PAIVS
3,<5,Normal,Normal,Normal,no,no,Normal,None,no,Normal,no,<7.5,no,<5,Severe,4-10_days,no,Equal,Transp.,TGA
4,5-12,Low,Oligaemic,Normal,no,no,Normal,Lt_to_Rt,yes,Normal,no,<7.5,yes,<5,Moderate,11-30_days,no,Equal,Complete,PAIVS


## 3. Split Train / Test

In [18]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, test_size=0.2, random_state=SEED)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f" Split réalisé")
print(f"   Train : {len(df_train)} obs  ({len(df_train)/N_SAMPLES:.0%})")
print(f"   Test  : {len(df_test)}  obs  ({len(df_test)/N_SAMPLES:.0%})")

 Split réalisé
   Train : 4000 obs  (80%)
   Test  : 1000  obs  (20%)


## 4. Apprentissage de Structure — HillClimb Search (BIC)

In [19]:
from pgmpy.estimators import HillClimbSearch, BIC

print("⏳ HillClimb (BIC) en cours...")
hc = HillClimbSearch(data=df_train)
dag_hc = hc.estimate(
    scoring_method=BIC(data=df_train),
    max_iter=500,
    show_progress=False
)
print(f"   Structure apprise : {len(dag_hc.edges())} arcs trouvés.")
print("   Arcs :", sorted(dag_hc.edges()))

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'LowerBodyO2': 'C', 'LungFlow': 'C', 'ChestXray': 'C', 'XrayReport': 'C', 'BirthAsphyxia': 'C', 'Grunting': 'C', 'LungParench': 'C', 'DuctFlow': 'C', 'LVH': 'C', 'CO2': 'C', 'Sick': 'C', 'CO2Report': 'C', 'LVHreport': 'C', 'RUQO2': 'C', 'HypoxiaInO2': 'C', 'Age': 'C', 'GruntingReport': 'C', 'HypDistrib': 'C', 'CardiacMixing': 'C', 'Disease': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'LowerBodyO2': 'C', 'LungFlow': 'C', 'ChestXray': 'C', 'XrayReport': 'C', 'BirthAsphyxia': 'C', 'Grunting': 'C', 'LungParench': 'C', 'DuctFlow': 'C', 'LVH': 'C', 'CO2': 'C', 'Sick': 'C', 'CO2Report': 'C', 'LVHreport': 'C', 'RUQO2': 'C', 'HypoxiaInO2': 'C', 'Age': 'C', 'GruntingReport': 'C', 'HypDistrib': 'C', 'CardiacMixing': 'C', 'Disease': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferre

⏳ HillClimb (BIC) en cours...
   Structure apprise : 29 arcs trouvés.
   Arcs : [('CO2', 'CO2Report'), ('CardiacMixing', 'HypDistrib'), ('CardiacMixing', 'HypoxiaInO2'), ('ChestXray', 'LungFlow'), ('ChestXray', 'LungParench'), ('Disease', 'Age'), ('Disease', 'BirthAsphyxia'), ('Disease', 'CardiacMixing'), ('Disease', 'DuctFlow'), ('Disease', 'Sick'), ('DuctFlow', 'HypDistrib'), ('Grunting', 'GruntingReport'), ('HypDistrib', 'LowerBodyO2'), ('HypoxiaInO2', 'LowerBodyO2'), ('HypoxiaInO2', 'RUQO2'), ('LVH', 'Disease'), ('LVHreport', 'LVH'), ('LungFlow', 'CardiacMixing'), ('LungFlow', 'Disease'), ('LungFlow', 'LVH'), ('LungFlow', 'LVHreport'), ('LungFlow', 'LungParench'), ('LungParench', 'CO2'), ('LungParench', 'Disease'), ('LungParench', 'Grunting'), ('LungParench', 'HypoxiaInO2'), ('Sick', 'Age'), ('Sick', 'Grunting'), ('XrayReport', 'ChestXray')]


## 5. Apprentissage des Paramètres & Évaluation

In [23]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.metrics import log_likelihood_score, structure_score

# Création du modèle avec la structure apprise
model_hc = DiscreteBayesianNetwork(dag_hc.edges())

# Apprentissage des paramètres (CPDs) sur les données de train
model_hc.fit(df_train, estimator=MaximumLikelihoodEstimator)
print("✅ Paramètres (CPDs) appris.")

# Évaluation sur les données de test
bic_score = structure_score(model_hc, df_test, scoring_method='bic-d')

print(f"\nMétriques du modèle HillClimb (sur Test) :")
print(f"   BIC Score       : {bic_score:.2f}")

# Accuracy MAP sur 'Disease'
def quick_accuracy(model, test_df, target='Disease', n=100):
    from pgmpy.inference import VariableElimination
    infer = VariableElimination(model)
    sample = test_df.sample(n, random_state=42)
    correct = 0
    for _, row in sample.iterrows():
        evid = {c: row[c] for c in test_df.columns if c != target and c in model.nodes()}
        pred = infer.map_query([target], evidence=evid, show_progress=False)[target]
        if pred == row[target]: correct += 1
    return correct / n

acc = quick_accuracy(model_hc, df_test)
print(f"   Accuracy MAP    : {acc:.1%}")

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'LowerBodyO2': 'C', 'LungFlow': 'C', 'ChestXray': 'C', 'XrayReport': 'C', 'BirthAsphyxia': 'C', 'Grunting': 'C', 'LungParench': 'C', 'DuctFlow': 'C', 'LVH': 'C', 'CO2': 'C', 'Sick': 'C', 'CO2Report': 'C', 'LVHreport': 'C', 'RUQO2': 'C', 'HypoxiaInO2': 'C', 'Age': 'C', 'GruntingReport': 'C', 'HypDistrib': 'C', 'CardiacMixing': 'C', 'Disease': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'LowerBodyO2': 'C', 'LungFlow': 'C', 'ChestXray': 'C', 'XrayReport': 'C', 'BirthAsphyxia': 'C', 'Grunting': 'C', 'LungParench': 'C', 'DuctFlow': 'C', 'LVH': 'C', 'CO2': 'C', 'Sick': 'C', 'CO2Report': 'C', 'LVHreport': 'C', 'RUQO2': 'C', 'HypoxiaInO2': 'C', 'Age': 'C', 'GruntingReport': 'C', 'HypDistrib': 'C', 'CardiacMixing': 'C', 'Disease': 'C'}


✅ Paramètres (CPDs) appris.

Métriques du modèle HillClimb (sur Test) :
   BIC Score       : -13164.15
   Accuracy MAP    : 85.0%
